# lOCAL OLLAMA TESTING WITH PDF

### Import Libraries


In [9]:
%pip install --q langchain langchain-community langchain-ollama chromadb pypdf
%pip install --q ipywidgets tqdm

Note: you may need to restart the kernel to use updated packages.


d:\3-2\Ollama\rag_project\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


d:\3-2\Ollama\rag_project\.venv\Scripts\python.exe: No module named pip


### Installation


In [10]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Jupyter-specific imports
from IPython.display import display, Markdown

# Set environment variable for protobuf
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

## Load PDF
Load the PDF document from the specified path.

In [11]:
# Load PDF
local_path = "../Data/PDF/WEF_The_Global_Cooperation_Barometer_2024.pdf"
if local_path:
    loader = PyPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"PDF loaded successfully: {local_path}")
    print(f"Number of pages: {len(data)}")
else:
    print("Upload a PDF file")

PDF loaded successfully: ../Data/PDF/WEF_The_Global_Cooperation_Barometer_2024.pdf
Number of pages: 26


In [13]:
# Preview first page
Markdown(data[1].page_content)

Contents
Images: Getty Images
© 2024 World Economic Forum. All rights 
reserved. No part of this publication may 
be reproduced or transmitted in any form 
or by any means, including photocopying 
and recording, or by any information 
storage and retrieval system.
Disclaimer 
This document is published by the  
World Economic Forum as a contribution 
to a project, insight area or interaction. 
The findings, interpretations and 
conclusions expressed herein are a result 
of a collaborative process facilitated and 
endorsed by the World Economic Forum 
but whose results do not necessarily 
represent the views of the World Economic 
Forum, nor the entirety of its Members, 
Partners or other stakeholders.
Foreword 
About the Global Cooperation Barometer
Executive summary 
Introduction: The state of global cooperation
Five pillars of global cooperation
Pillar 1 Trade and capital 
Pillar 2 Innovation and technology
Pillar 3 Climate and natural capital
Pillar 4 Health and wellness
Pillar 5 Peace and security
Conclusion: Towards a more cooperative future 
Appendix: Sources and methodology
Contributors
Endnotes
3
4
6
7
9
9
11
13
15
17
19
20
22
23
The Global Cooperation Barometer 2024
2

## set up llm and retrieval

In [ ]:
# Set up LLM and retrieval
local_model = "mistral"  # or whichever model you prefer
llm = ChatOllama(model=local_model)

In [ ]:
# Query prompt template
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 2
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}""",
)

In [ ]:
# Set up retriever
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(), 
    llm,
    prompt=QUERY_PROMPT
)

## chain creation

In [ ]:
# Set up retriever
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(), 
    llm,
    prompt=QUERY_PROMPT
)

In [ ]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## pdf chatting

In [ ]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    return display(Markdown(chain.invoke(question)))

In [ ]:
# Example 1
chat_with_pdf("What is the main idea of this document?")

In [ ]:
# Example 2
chat_with_pdf("Can you explain the case study highlighted in the document?")

# Trying system requirement model 

In [14]:
!ollama list

NAME                       ID              SIZE      MODIFIED   
mistral:latest             6577803aa9a0    4.4 GB    4 days ago    
nomic-embed-text:latest    0a109f422b47    274 MB    4 days ago    


In [ ]:
!ollama pull qwen2:7b

In [ ]:
!ollama list

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2:3b",
    temperature=0.2
)

response = llm.invoke("বাংলায় বলো: Artificial Intelligence কী?")
print(response.content)

In [ ]:
response = llm.invoke("Explain machine learning in simple English.")
print(response.content)

In [ ]:
!ollama pull mxbai-embed-large

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="mxbai-embed-large"
)

vector = embeddings.embed_query("বাংলাদেশ একটি সুন্দর দেশ")
print(len(vector))

# Mini Rag test

In [ ]:
from langchain_community.vectorstores import Chroma

texts = [
    "বাংলাদেশ দক্ষিণ এশিয়ার একটি দেশ।",
    "Artificial Intelligence is transforming the world."
]

vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

docs = retriever.invoke("বাংলাদেশ কোথায় অবস্থিত?")
print(docs)